In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle
import glob

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "ERA5_Data")
dataType = "ERA5Comparison_AreaAverage"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

# spinup_hours = "24"
# spinup_hours = "12"
# spinup_hours = "6"
spinup_hours = "0"

RunType = ("TRACER","MOIST","NSSL",spinup_hours)
# RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

RunType = ("TRACER","MOIST","TEMPO",spinup_hours)
# RunType = ("TRACER","DRY","TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing Plotting Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_Plotting import ContourPlotting_Class

In [ ]:
#Importing DataSaving Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetNumElements():
    num_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return num_elements
loop_elements = GetNumElements()

In [ ]:
########################
#LOADING DATA

In [ ]:
# ERA5_VariableName_List = ['crwc', 
#  't', #* #need to convert model to temperature
#  'u', 'v', 'q', 'w', 
#  # 'vo', #* not added to convert_mpas yet
#  'd', 'r', 'clwc', 'ciwc']
# ERA5_VariableName_List

In [ ]:
def ConvertPressureToHeight_ERA5(pressureArray, temperatureArray, qvArray,
                                 R_d=287.0, g=9.81):
    """
    Hypsometric equation for ERA5 using T and q (3D fields).
    """

    pressureArray = np.asarray(pressureArray, dtype=float)

    # Convert hPa → Pa if needed
    if np.nanmean(pressureArray) < 2000:
        pressureArray *= 100.0

    # Compute virtual temperature
    virtualTemperature = temperatureArray * (1.0 + 0.61 * qvArray)

    nz, ny, nx = virtualTemperature.shape

    heightArray = np.zeros((nz, ny, nx))

    for k in range(1, nz):
        p1 = pressureArray[k-1]
        p2 = pressureArray[k]

        Tv_layer = 0.5 * (virtualTemperature[k-1] + virtualTemperature[k])
        deltaZ = (R_d * Tv_layer / g) * np.log(p1 / p2)

        heightArray[k] = heightArray[k-1] + deltaZ

    return heightArray


In [ ]:
# virtualTemperature = temperature * (1.0 + 0.61 * qv)
# virtualTemperature.data

In [ ]:
pressure = ERA5_t_subset.level
temperature = ERA5DataLoading_Class.LoadERA5Data_gdex(DirectoryManager, ModelData_NSSL, timeString,"t")
qv = ERA5DataLoading_Class.LoadERA5Data_gdex(DirectoryManager, ModelData_NSSL, timeString,"q")

heightArray = ConvertPressureToHeight_ERA5(pressureArray=pressure, temperatureArray=temperature, qvArray=qv)

In [ ]:
heightArray[:,0,0]